In [ ]:
#Importation
import math
import numpy as np
import numpy as np
from math import sin, cos, atan2
import matplotlib.pyplot as plt

#global variable
ROT_SPEED_TO_M_PER_S = 0.0003375  # needs to be calibrated
WHEEL_BASE = 0.095 # needs to be calibrated


#---------main---------

#inizialisation 

rot_speed_left_motor = 0
rot_speed_right_motor = 0

#initialisation for first robot position
cv_out = None
while cv_out is None:                       # wait until the robot is seen
    cv_out = fetch_status_cv(get_image())
X, Y, theta = cv_out[:3]
next_point = cv_out[3]
P = R.copy()
t_prev = time.time()

while True:
    now = time.time()
    dt = now - t_prev
    t_prev = now

    speed_left, speed_right = read_wheel_speeds()    # measured, not commanded

    cv_out = fetch_status_cv(get_image())
    if cv_out is None:
        camera_pose = None                           # keep the previous next_point
    else:
        camera_pose = np.array(cv_out[:3])
        next_point = cv_out[3]

    X, Y, theta, P = kalman(X, Y, theta, P, speed_left, speed_right, dt, camera_pose)

    go_to_next_point(X, Y, theta, next_point)


def wrap_to_pi(a):
    return atan2(sin(a), cos(a))

                     
def odometry(rot_speed_left_motor, rot_speed_right_motor, X_position, Y_position, angle_position_rad, delta_time):
    
    #Linear speed
    linear_speed_left = rot_speed_left_motor * ROT_SPEED_TO_M_PER_S
    linear_speed_right = rot_speed_right_motor * ROT_SPEED_TO_M_PER_S
    linear_speed = (linear_speed_right + linear_speed_left) / 2 
    
    #Angular speed
    angular_speed = (linear_speed_right - linear_speed_left) / WHEEL_BASE  # rad/s

    angle_mid = angle_position_rad + angular_speed * delta_time / 2 #middle point to make an etimation of 2 segment
     
    X_new = X_position + linear_speed * math.cos(angle_mid) * delta_time
    Y_new = Y_position + linear_speed * math.sin(angle_mid) * delta_time
    angle_new = angle_position_rad + angular_speed * delta_time

    return X_new, Y_new, angle_new                    


def kalman(X, Y, theta, P, speed_left, speed_right, dt):

    # ---------- PREDICTION ----------
    X_pred, Y_pred, theta_pred = odometry(speed_left, speed_right, X, Y, theta, dt)
    theta_pred = wrap_to_pi(theta_pred)
    state_pred = np.array([X_pred, Y_pred, theta_pred])

    v = (speed_right + speed_left) * ROT_SPEED_TO_M_PER_S / 2
    omega = (speed_right - speed_left) * ROT_SPEED_TO_M_PER_S / WHEEL_BASE
    theta_mid = theta + omega * dt / 2

    F = np.array([[1, 0, -v * sin(theta_mid) * dt],
                  [0, 1,  v * cos(theta_mid) * dt],
                  [0, 0,  1]])

    q_xy = SIGMA_D2 * abs(v * dt) + Q_MIN
    q_th = SIGMA_TH2 * abs(omega * dt) + Q_MIN
    Q = np.diag([q_xy, q_xy, q_th])

    P_pred = F @ P @ F.T + Q

    # ---------- CAMERA ----------
    cv_output = fetch_status_cv()          # (X_cv, Y_cv, theta_cv, next_position) or None

    if cv_output is None:
        return X_pred, Y_pred, theta_pred, P_pred      # odometry only

    camera_pose = np.array(cv_output[:3])              # keep only X, Y, theta

    # ---------- UPDATE ----------
    innovation = np.asarray(camera_pose) - H @ state_pred
    innovation[2] = wrap_to_pi(innovation[2])

    S = H @ P_pred @ H.T + R
    K = P_pred @ H.T @ np.linalg.inv(S)

    state_new = state_pred + K @ innovation
    theta_new = wrap_to_pi(state_new[2])
    P_new = (np.eye(3) - K @ H) @ P_pred

    return state_new[0], state_new[1], theta_new, P_new    